<a href="https://colab.research.google.com/github/Mathildeholst/Speciale/blob/main/Stemmer_fake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/syv-ai/plapre.git
!pip install --upgrade "transformers>=4.51" "tokenizers>=0.21"

In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [1]:
from huggingface_hub import snapshot_download
import json, os

model_dir = snapshot_download("syvai/plapre-nano")
config_path = os.path.join(model_dir, "tokenizer_config.json")
with open(config_path) as f:
    config = json.load(f)

config["tokenizer_class"] = "PreTrainedTokenizerFast"
config["extra_special_tokens"] = {}  # fix: transformers expects a dict, not a list

with open(config_path, "w") as f:
    json.dump(config, f)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [2]:
import json, torch, numpy as np, soundfile as sf
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn as nn
from huggingface_hub import hf_hub_download
from kanade_tokenizer import KanadeModel, load_vocoder, vocode

tokenizer = AutoTokenizer.from_pretrained("syvai/plapre-nano")
model = AutoModelForCausalLM.from_pretrained(
    "syvai/plapre-nano", dtype=torch.bfloat16,
    device_map="auto", ignore_mismatched_sizes=True)

proj_path = hf_hub_download("syvai/plapre-nano", "speaker_proj.pt")
speaker_proj = nn.Linear(128, 960)
speaker_proj.load_state_dict(torch.load(proj_path, map_location="cpu"))
speaker_proj = speaker_proj.to(torch.bfloat16).eval()

import plapre
with open(Path(plapre.__file__).parent / "speakers.json") as f:
    speakers_raw = json.load(f)
speakers = {k: torch.tensor(v, dtype=torch.float32) for k, v in speakers_raw.items()}

device = torch.device("cuda")
kanade = KanadeModel.from_pretrained("frothywater/kanade-25hz-clean").eval().to(device)
vocoder_model = load_vocoder(kanade.config.vocoder_name).to(device)

def speak(text, speaker_name=None, temperature=0.8, top_p=0.95, top_k=50, max_new_tokens=500):
    spk_name = speaker_name or list(speakers.keys())[0]
    spk_emb = speakers[spk_name].to(device)
    with torch.no_grad():
        spk_hidden = speaker_proj(spk_emb.cpu().to(torch.bfloat16)).unsqueeze(0).unsqueeze(0)
    text_tag = tokenizer.convert_tokens_to_ids("<text>")
    audio_tag = tokenizer.convert_tokens_to_ids("<audio>")
    text_ids = tokenizer.encode(text, add_special_tokens=False)
    prompt_ids = [text_tag] + text_ids + [audio_tag]
    input_ids = torch.tensor([prompt_ids], device=device)
    with torch.no_grad():
        token_embeds = model.model.embed_tokens(input_ids)
        full_embeds = torch.cat([spk_hidden.to(device), token_embeds], dim=1)
    out = model.generate(
        inputs_embeds=full_embeds, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature,
        top_p=top_p, top_k=top_k, eos_token_id=tokenizer.eos_token_id)
    audio_start = tokenizer.convert_tokens_to_ids("<audio_0>")
    audio_end = tokenizer.convert_tokens_to_ids("<audio_12799>")
    generated = out[0].tolist()
    kanade_indices = [t - audio_start for t in generated if audio_start <= t <= audio_end]
    if not kanade_indices:
        print("Ingen audio-tokens genereret!")
        return None
    tokens_tensor = torch.tensor(kanade_indices, dtype=torch.long, device=device)
    with torch.no_grad():
        mel = kanade.decode(content_token_indices=tokens_tensor, global_embedding=spk_emb.float().to(device))
        waveform = vocode(vocoder_model, mel.unsqueeze(0))
    audio = waveform.squeeze().cpu().numpy()
    sf.write("output.wav", audio, 24000)
    return audio


[2026-03-23 08:36:50,825] WARNING kanade_tokenizer: FlashAttention is not installed. Falling back to PyTorch SDPA implementation. There is no warranty that the model will work correctly.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

config.yaml: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/563M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/torchaudio/models/wavlm_base_plus.pth" to /root/.cache/torch/hub/checkpoints/wavlm_base_plus.pth


100%|██████████| 360M/360M [00:01<00:00, 369MB/s]
[2026-03-23 08:37:03,650] INFO kanade_tokenizer: Loaded weights from safetensors file: /root/.cache/huggingface/hub/models--frothywater--kanade-25hz-clean/snapshots/cb2c8f10959ff0e5d3e97c9b82fcc3779c9532a5/model.safetensors
INFO:kanade_tokenizer:Loaded weights from safetensors file: /root/.cache/huggingface/hub/models--frothywater--kanade-25hz-clean/snapshots/cb2c8f10959ff0e5d3e97c9b82fcc3779c9532a5/model.safetensors


hift.pt:   0%|          | 0.00/83.4M [00:00<?, ?B/s]

In [3]:
audio = speak("Hej, hvordan har du det?")

from IPython.display import Audio
Audio("output.wav")